# Component 1: DeBERTa-v3 & RoBERTa Transformer Fine-Tuning for Hospitality ABSA
**Project ID:** J26-DS-345  
**Component:** 1 – Trust-Aware Review Analysis & Aspect Sentiment Engine  
**Candidate:** DISSANAYAKE L.G.D.P.M. (IT23252554)  
**Supervisors:** Prof. Samantha Thelijjagoda, Dr. Junius Anjana  
**Degree:** B.Sc. (Hons) in Information Technology Specializing in Data Science, SLIIT  

---
### Notebook Purpose
This interactive notebook fine-tunes Transformer models (**DeBERTa-v3-small** and **RoBERTa-base**) on the curated 400-sentence Sri Lankan hospitality ground-truth dataset.
It conditions sentiment classification on the target aspect (*Cleanliness, Food, Staff, Room, Location, Wi-Fi*) to classify 3-class polarity:
- **0: Negative**
- **1: Neutral**
- **2: Positive**

> **Tip for Google Colab:** Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` (or A100 in Colab Pro) for fast hardware acceleration.

In [ ]:
# 1. Install Required Libraries
!pip install -q transformers datasets accelerate torch evaluate scikit-learn matplotlib seaborn

In [ ]:
# 2. Verify GPU Acceleration
import torch
print("PyTorch Version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if device == "cuda":
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("VRAM Available:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("[!] Running on CPU. Training will be slower. Enable GPU in Colab Runtime settings.")

### Step 1: Load Ground-Truth Benchmark & Hard Linguistic Challenges
We load both the stratified 400-sentence standard benchmark and the 50 hard linguistic challenges (sarcasm, implicit sentiment, double negation, Sri Lankan vernacular) to form a unified 450-sentence gold-standard training corpus.

In [ ]:
import os
import pandas as pd

LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}

# Find paths (works in local IDE and in Colab)
std_path = "data/processed/ground_truth_annotated_hospitality.csv"
if not os.path.exists(std_path):
    std_path = "ground_truth_annotated_hospitality.csv"

hard_path = "data/processed/hard_linguistic_challenges.csv"
if not os.path.exists(hard_path):
    hard_path = "hard_linguistic_challenges.csv"

records = []
if os.path.exists(std_path):
    df_std = pd.read_csv(std_path)
    for _, row in df_std.iterrows():
        s = str(row.get("sentiment_label", "")).strip()
        if s in LABEL2ID:
            records.append({
                "aspect": str(row["aspect_category"]).lower().strip(),
                "sentence": str(row["review_sentence"]).strip(),
                "sentiment": s,
                "label": LABEL2ID[s],
                "source": "Standard Benchmark"
            })

if os.path.exists(hard_path):
    df_hard = pd.read_csv(hard_path)
    for _, row in df_hard.iterrows():
        s = str(row.get("ground_truth_sentiment", "")).strip()
        if s in LABEL2ID:
            p_type = row.get("phenomenon_type", "")
            records.append({
                "aspect": str(row["target_aspect"]).lower().strip(),
                "sentence": str(row["review_sentence"]).strip(),
                "sentiment": s,
                "label": LABEL2ID[s],
                "source": f"Hard ({p_type})"
            })

df_unified = pd.DataFrame(records)
print(f"[+] Loaded {len(df_unified)} total annotated sentences across all categories.")
print(f"    Standard samples : {sum(df_unified['source'] == 'Standard Benchmark')}")
print(f"    Hard challenges  : {sum(df_unified['source'] != 'Standard Benchmark')}")
print("\nSentiment Distribution:")
print(df_unified["sentiment"].value_counts())
display(df_unified.head(5))


### Step 2: Format Inputs for Aspect-Conditioned Classification
We format each training instance as:
`Aspect: [category] | Sentence: [review_sentence]`
This teaches the Transformer attention mechanism to focus specifically on the target aspect.

In [ ]:
from sklearn.model_selection import train_test_split

df_unified["input_text"] = "Aspect: " + df_unified["aspect"] + " | Sentence: " + df_unified["sentence"]

train_df, val_df = train_test_split(
    df_unified,
    test_size=0.20,
    random_state=42,
    stratify=df_unified["label"]
)
print(f"Training set size   : {len(train_df)} sentences (80 percent)")
print(f"Validation set size : {len(val_df)} sentences (20 percent)")


### Step 3: Tokenization & PyTorch Dataset Formulation
We use Hugging Face `AutoTokenizer` configured with `max_length=128`.

In [ ]:
from transformers import AutoTokenizer
import torch

MODEL_NAME = "microsoft/deberta-v3-small"  # Alternative: "roberta-base"
print(f"Loading tokenizer for: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(texts):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_enc = tokenize_batch(train_df["input_text"])
val_enc = tokenize_batch(val_df["input_text"])

class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ABSADataset(train_enc, train_df["label"].values)
val_dataset = ABSADataset(val_enc, val_df["label"].values)
print("Datasets successfully assembled.")

### Step 4: Model Architecture & Training
We load `AutoModelForSequenceClassification` with 3 output heads and fine-tune using Hugging Face `Trainer`.

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "macro_precision": float(precision),
        "macro_recall": float(recall)
    }

OUTPUT_DIR = "models/deberta_absa_finetuned"
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Starting Transformer fine-tuning...")
trainer.train()

### Step 5: Validation Evaluation & Confusion Matrix
We evaluate the best checkpoint on the held-out validation set and plot the confusion matrix.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

eval_results = trainer.evaluate()
print("\n" + "=" * 50)
print("   VALIDATION EVALUATION METRICS")
print("=" * 50)
for k, v in eval_results.items():
    print(f"{k:20s}: {v}")

# Compute predictions
raw_preds = trainer.predict(val_dataset)
y_pred = np.argmax(raw_preds.predictions, axis=1)
y_true = val_df["label"].values

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Negative", "Neutral", "Positive"]))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Neutral", "Positive"],
            yticklabels=["Negative", "Neutral", "Positive"])
plt.title(f"{MODEL_NAME} Aspect Sentiment Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Ground Truth Sentiment")
plt.tight_layout()
plt.show()

### Step 6: Interactive Sentiment Prediction Demo
Test individual Sri Lankan hospitality sentences with custom aspect queries.

In [ ]:
def predict_sentiment(aspect: str, sentence: str):
    query = f"Aspect: {aspect} | Sentence: {sentence}"
    inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
    return ID2LABEL[pred_id], {ID2LABEL[i]: round(float(probs[i]), 4) for i in range(3)}

test_samples = [
    ("food", "The breakfast buffet had delicious egg hoppers and fresh tropical fruits."),
    ("cleanliness", "The bathroom was disgusting with cockroaches and stained towels."),
    ("wifi", "Internet connection was okay for checking emails but kept disconnecting during Zoom calls."),
    ("staff", "Manager Anura was exceptionally kind and arranged our early morning safari jeep.")
]

print("\n--- Interactive Predictions ---")
for asp, sent in test_samples:
    label, probs = predict_sentiment(asp, sent)
    print(f"Aspect  : {asp}")
    print(f"Sentence: {sent}")
    print(f"Result  : {label.upper()} - Probabilities: {probs}\n")

### Step 7: Export to ONNX Runtime for Low-Latency Microservice Inference
To meet Non-Functional Requirement 3 (API latency <= 200 ms) in FastAPI (`src/main.py`), we export the fine-tuned DeBERTa model to ONNX format for fast CPU evaluation.

In [ ]:
# Export fine-tuned PyTorch model to ONNX
import os
import torch.onnx

onnx_output_path = "models/deberta_v3_small_absa.onnx"
os.makedirs("models", exist_ok=True)

dummy_input = tokenizer("Aspect: room | Sentence: The air conditioner was very quiet", return_tensors="pt", max_length=128, padding="max_length", truncation=True)
dummy_input = {k: v.to(device) for k, v in dummy_input.items()}

model.eval()
try:
    torch.onnx.export(
        model,
        (dummy_input["input_ids"], dummy_input["attention_mask"]),
        onnx_output_path,
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={"input_ids": {0: "batch_size"}, "attention_mask": {0: "batch_size"}, "logits": {0: "batch_size"}},
        opset_version=14
    )
    print(f"[+] Model successfully exported to ONNX format at: {onnx_output_path}")
except Exception as e:
    print(f"[!] ONNX export note: {e}")
